# Galerie GenAI — 04 · Fine-tuning LoRA : spécialiser un petit modèle 🟠

> ⚠️ **À faire APRÈS la remise de ton cas d'usage certif.** Le fine-tuning est
> le **dernier** recours, pas le premier réflexe — et il n'a aucune place dans
> le cas d'usage certif (ML classique).
>
> **⚙️ Ce notebook exige un GPU → lance-le sur [Google Colab](https://colab.research.google.com)**
> (`Exécution › Modifier le type d'exécution › T4 GPU`, gratuit). Il ne tourne
> pas sur un CPU de portable en temps raisonnable.
>
> **Étagère optionnelle** — pas un brief, pas de livrable, pas de note.
> **Durée** : ~3 h (dont ~20-30 min d'entraînement sur T4).
> **Fiches** : `panorama_genai_llm_rag_agents.md` · `cheatsheet_sobriete_couts.md`
> · galerie GenAI 01-02 (RAG) comme point de comparaison.

## Quand fine-tuner ? (lis ça AVANT de lancer quoi que ce soit)

Trois leviers pour adapter un LLM à ton besoin, **du moins cher au plus cher** :

| Levier | Coût | Ce qu'il change | Quand |
|---|---|---|---|
| **Prompt** (instructions, few-shot) | ~0 | le comportement immédiat | toujours essayer en premier |
| **RAG** (cf. notebooks 01-02) | modéré | la **connaissance** accessible | l'info vit dans des documents qui bougent |
| **Fine-tuning** (ce notebook) | élevé (GPU, données, MLOps) | le **style, le format, le comportement** | prompt + RAG ne suffisent pas, et tu as des exemples |

> 🧭 Règle d'or : **le fine-tuning n'apprend pas des faits, il apprend un
> comportement.** Pour « connaître le catalogue produit » → RAG. Pour
> « répondre TOUJOURS dans CE format JSON métier » → fine-tuning. Se tromper
> de levier coûte cher pour rien.

## Le cas — un format de sortie métier

**NovaThread** veut trier ses tickets SAV automatiquement : chaque ticket →
un JSON `{categorie, urgence, resume}`. Tu vas voir dans un instant qu'un
petit modèle sait déjà produire du JSON et deviner la catégorie… mais qu'il
**rate la notion d'urgence**, parce que « urgence » est une **règle interne
NovaThread** qu'aucun modèle ne peut deviner. C'est le cas d'école du
fine-tuning léger : lui enseigner CE comportement métier précis.

## Setup — GPU + dépendances (versions figées)

L'écosystème fine-tuning bouge très vite : on **fige les versions** testées
pour ce notebook. Si une cellule casse sur Colab, c'est presque toujours une
version qui a bougé — recolle ces pins.

In [ ]:
# Sur Colab, décommente la ligne d'installation (elle prend ~2 min).
# !pip install -q "transformers==5.13.1" "trl==1.8.0" "peft==0.19.1" "datasets==3.6.0" "accelerate>=1.0"

import torch

assert torch.cuda.is_available(), (
    "Aucun GPU détecté. Sur Colab : Exécution › Modifier le type d'exécution › T4 GPU. "
    "Ce notebook n'est pas conçu pour tourner sur CPU.")
print("GPU :", torch.cuda.get_device_name(0))

## [1] Le dataset — tickets SAV → JSON

Fourni dans `finetuning_data/` : 273 exemples d'entraînement au **format chat**
(system + user + assistant) et 60 tickets de test avec leur sortie attendue.
Génération reproductible (`build_dataset.py`, `random_state=42`).

Sur Colab, téléverse le dossier `finetuning_data/` ou clone le repo. Le
`instruction.txt` contient la consigne système, identique à l'entraînement et
à l'évaluation — **cohérence prompt = condition n°1 d'un bon fine-tuning**.

In [ ]:
import json
from pathlib import Path

DATA = Path("finetuning_data")
INSTRUCTION = (DATA / "instruction.txt").read_text(encoding="utf-8")
train_chat = [json.loads(l) for l in (DATA / "train.jsonl").read_text(encoding="utf-8").splitlines()]
test = [json.loads(l) for l in (DATA / "test.jsonl").read_text(encoding="utf-8").splitlines()]

print(f"train : {len(train_chat)} exemples | test : {len(test)} exemples")
print("\nInstruction système :\n", INSTRUCTION)
print("\nExemple d'entraînement (rôles) :", [m["role"] for m in train_chat[0]["messages"]])
print("Sortie attendue visée :", train_chat[0]["messages"][-1]["content"])

## [2] Le modèle de base et sa mesure AVANT — la ligne de départ

On charge **Qwen2.5-0.5B-Instruct** (petit, tient large sur T4). On mesure
son comportement AVANT tout fine-tuning, avec un harnais d'éval sévère : JSON
parsable ? schéma respecté (Pydantic) ? catégorie exacte ? **urgence exacte** ?

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELE_BASE = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODELE_BASE)
modele_base = AutoModelForCausalLM.from_pretrained(MODELE_BASE, dtype=torch.bfloat16, device_map="auto")
print("modèle chargé :", sum(p.numel() for p in modele_base.parameters()) // 1_000_000, "M paramètres")


def genere(modele, ticket: str, max_new_tokens: int = 80) -> str:
    messages = [{"role": "system", "content": INSTRUCTION},
                {"role": "user", "content": ticket}]
    entree = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True).to(modele.device)
    with torch.no_grad():
        sortie = modele.generate(**entree, max_new_tokens=max_new_tokens, do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(sortie[0][entree["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
import re
from typing import Literal

from pydantic import BaseModel, ValidationError


class SortieTicket(BaseModel):
    categorie: Literal["livraison", "produit_defectueux", "remboursement",
                       "taille_ou_ajustement", "paiement", "compte_client", "autre"]
    urgence: Literal["basse", "moyenne", "haute"]
    resume: str


def extrait_json(texte: str):
    """Parsing robuste : JSON direct, sinon 1er objet {...} trouvé (gère les ```json … ```)."""
    try:
        return json.loads(texte)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", texte, re.DOTALL)
        if not m:
            return None
        try:
            return json.loads(m.group(0))
        except json.JSONDecodeError:
            return None


def evalue(modele, jeu_test: list) -> dict:
    n = len(jeu_test)
    json_ok = schema_ok = cat_ok = urg_ok = 0
    for ex in jeu_test:
        obj = extrait_json(genere(modele, ex["ticket"]))
        if obj is None:
            continue
        json_ok += 1
        try:
            valide = SortieTicket(**obj)
        except ValidationError:
            continue
        schema_ok += 1
        cat_ok += valide.categorie == ex["sortie"]["categorie"]
        urg_ok += valide.urgence == ex["sortie"]["urgence"]
    return {"json_parsable_%": round(100 * json_ok / n), "schema_valide_%": round(100 * schema_ok / n),
            "categorie_exacte_%": round(100 * cat_ok / n), "urgence_exacte_%": round(100 * urg_ok / n)}


score_avant = evalue(modele_base, test)
print("AVANT fine-tuning :", score_avant)

> 🧭 **Lis le profil, pas juste la moyenne.** Tu verras typiquement un modèle
> déjà correct sur le format JSON et la catégorie, mais **faible sur
> l'urgence** — la seule dimension qui encode une règle métier NovaThread. Un
> petit modèle non spécialisé n'a aucun moyen de la deviner. **C'est
> exactement ce que le fine-tuning va lui apprendre** — et pourquoi on mesure
> AVANT : sans point de départ, « après » ne veut rien dire.

## [3] Le fine-tuning LoRA

**LoRA** (*Low-Rank Adaptation*) n'entraîne pas les 500 M de paramètres : il
ajoute de petites matrices (quelques M de paramètres) et ne bouge qu'elles.
Résultat : entraînement rapide, faible mémoire, et un « adaptateur » de
quelques Mo au lieu d'un modèle complet. C'est le standard du fine-tuning
sobre — droit dans la ligne du parcours.

In [ ]:
from datasets import load_dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

dataset_train = load_dataset("json", data_files=str(DATA / "train.jsonl"), split="train")

config_lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # les projections d'attention
    task_type="CAUSAL_LM",
)

config_sft = SFTConfig(
    output_dir="qwen-novathread-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    max_length=512,
    bf16=True,
    report_to=[],           # pas de wandb ; mets "mlflow" si tu veux tracer (cf. M5-B2)
)

trainer = SFTTrainer(
    model=modele_base,
    train_dataset=dataset_train,
    peft_config=config_lora,
    args=config_sft,
)
trainer.train()   # ~20-30 min sur T4
print("entraînement terminé.")

## [4] La mesure APRÈS — le verdict

On recharge le modèle de base **propre** et on lui greffe l'adaptateur LoRA
entraîné, puis on ré-évalue sur le **même jeu de test**. C'est le geste que
tu connais depuis M1 : comparer avant/après sur une référence figée.

In [ ]:
from peft import PeftModel

base_propre = AutoModelForCausalLM.from_pretrained(MODELE_BASE, dtype=torch.bfloat16, device_map="auto")
modele_ft = PeftModel.from_pretrained(base_propre, "qwen-novathread-lora")

score_apres = evalue(modele_ft, test)

import pandas as pd
comparaison = pd.DataFrame({"avant (base)": score_avant, "après (LoRA)": score_apres})
comparaison["gain"] = comparaison["après (LoRA)"] - comparaison["avant (base)"]
comparaison

### Comment lire ce tableau

- Le gain le plus fort doit être sur **`urgence_exacte_%`** : c'est la règle
  métier que le modèle ne pouvait pas deviner et que les 273 exemples lui ont
  apprise. Si l'urgence ne bouge pas, vérifie que l'instruction système est
  **identique** entre entraînement et éval (cause n°1 d'un fine-tuning qui
  « ne prend pas »).
- Le format JSON et la catégorie peuvent déjà être hauts avant → peu de marge
  de gain, c'est normal (le fine-tuning ne « répare » que ce qui manquait).
- ⚠️ **Piège du sur-apprentissage** : avec seulement 273 exemples et 3
  époques, surveille que le modèle n'apprend pas par cœur. Le vrai juge est le
  **jeu de test** (jamais vu à l'entraînement) — c'est pour ça qu'on l'a mis
  de côté dès la génération du dataset.

## [5] ⭐ Ramener le modèle sur ta stack : export GGUF → Ollama

Boucler la boucle : ton modèle spécialisé doit tourner **en local**, comme
les modèles des notebooks 01-03. On fusionne l'adaptateur dans le modèle, on
convertit au format **GGUF** (celui d'Ollama/llama.cpp), et on l'importe.

> ⚠️ Section **⭐ optionnelle et fragile** : la conversion GGUF dépend de
> `llama.cpp` (versions sensibles). Ces commandes sont données à titre de
> chemin type ; attends-toi à ajuster selon les versions du jour.

In [ ]:
# 1) Fusionner l'adaptateur LoRA dans le modèle de base -> un modèle complet
modele_fusionne = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(MODELE_BASE, dtype=torch.bfloat16),
    "qwen-novathread-lora").merge_and_unload()
modele_fusionne.save_pretrained("qwen-novathread-fusionne")
tokenizer.save_pretrained("qwen-novathread-fusionne")
print("modèle fusionné sauvegardé.")

In [ ]:
# 2) Conversion GGUF via llama.cpp (à lancer dans un terminal / cellules shell Colab)
#    (⭐ non garanti selon les versions — c'est le maillon le plus fragile)
#
# !git clone https://github.com/ggerganov/llama.cpp
# !pip install -q -r llama.cpp/requirements.txt
# !python llama.cpp/convert_hf_to_gguf.py qwen-novathread-fusionne \
#         --outfile novathread.gguf --outtype q8_0
#
# 3) Import dans Ollama (sur ta machine, une fois novathread.gguf récupéré) :
#    - crée un fichier `Modelfile` contenant :
#        FROM ./novathread.gguf
#        SYSTEM """<recopie ici le contenu de instruction.txt>"""
#    - puis :  ollama create novathread -f Modelfile
#              ollama run novathread "Bonjour, colis jamais reçu, c'est urgent !"
print("Chemin d'export documenté ci-dessus (à exécuter hors CI).")

## 🔎 Ce que tu viens de pratiquer

- **Choisir le bon levier** : prompt < RAG < fine-tuning ; fine-tuner
  n'apprend pas des faits mais un **comportement / format**.
- **Mesurer avant / après** sur un jeu de test figé — le fine-tuning ne se
  juge JAMAIS « à l'œil ».
- **LoRA** : spécialiser en n'entraînant que quelques M de paramètres, un
  adaptateur de quelques Mo (sobriété).
- **La cohérence du prompt** entre entraînement et inférence : condition n°1.
- **Boucler sur ta stack locale** : merge → GGUF → Ollama.

## Ce qu'on n'a PAS fait (et qu'on assume)

- **Full fine-tuning** (tous les poids) : inutile ici, coûteux, et efface les
  capacités générales du modèle. LoRA suffit pour un format métier.
- **RLHF / DPO** (alignement par préférences) : métier de labo, hors périmètre
  intégrateur — à connaître de nom seulement.
- **Apprendre de la connaissance** par fine-tuning : mauvais levier → c'est le
  RAG (notebooks 01-02). Si tu voulais « connaître le catalogue », tu t'es
  trompé de notebook 🙂.

## ⭐ Pour aller plus loin (optionnel)

- Fais varier `r` (8, 16, 32) et le nombre d'époques : trace l'urgence sur le
  test — où apparaît le sur-apprentissage ?
- Ajoute une 4ᵉ dimension métier à prédire (ex. `produit_concerné`) : quelques
  lignes dans `build_dataset.py`, et re-mesure.
- Trace l'entraînement dans **MLflow** (`report_to="mlflow"`, cf. M5-B2) :
  params LoRA, loss, scores avant/après — le réflexe traçabilité, version LLM.
- Compare **coût/bénéfice** vs un simple `format="json"` forcé + few-shot dans
  le prompt (cf. notebook 01) : le fine-tuning valait-il le GPU sur CE cas ?